In [ ]:
import  lightkurve  as  lk 
import numpy as np 
import matplotlib.pyplot as plt 
%matplotlib inline
import astropy

In [ ]:

print("numpy:", np.__version__)
print("lightkurve:", lk.__version__)
print("astropy:", astropy.__version__)


In [ ]:
search_lc = lk.search_tesscut("tic 50897998")
search_lc

In [ ]:
S32 = search_lc[1].download(cutout_size=50)
S32.plot()

In [ ]:
print(S32.shape)

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

aper_new = np.zeros((50, 50), dtype=int)
aper_new[22:28, 22:28] = 1

aper_new[22:28, 22:28] = True
aper_big = np.zeros((50, 50), dtype=bool)
aper_big[21:29, 21:29] = True

S32.plot(aperture_mask=aper_new, mask_color='red')
plt.show()

lc_small = S32.to_lightcurve(aperture_mask=aper_new)
lc_big   = S32.to_lightcurve(aperture_mask=aper_big)
print("STD chica:", np.nanstd(lc_small.flux))
print("STD grande:", np.nanstd(lc_big.flux))


print("Flujo mediano chica:", np.nanmedian(lc_small.flux))
print("Flujo mediano grande:", np.nanmedian(lc_big.flux))


In [ ]:
lc_S32 = S32.to_lightcurve(aperture_mask='threshold')
lc_S32.plot(label="S32 FFI lc")

In [ ]:
from lightkurve.correctors import CBVCorrector
cbvCorrector = CBVCorrector(lc_S32, interpolate_cbvs=True)
cbvCorrector.cbvs

In [ ]:
cbvCorrector.cbvs[0].plot()

In [ ]:
cbv_type = ['MultiScale.1', 'MultiScale.2', 'MultiScale.3','Spike']

In [ ]:
cbv_indices = [np.arange(1,9), np.arange(1,9), np.arange(1,9), 'ALL']

In [ ]:
cbvCorrector.goodness_metric_scan_plot(cbv_type=cbv_type, cbv_indices=cbv_indices);
plt.plot([1.5e-1, 1.5e-1], [0,1])

In [ ]:
cbvcorrector = CBVCorrector(lc_S32, interpolate_cbvs=True)
cbvcorrector.correct_gaussian_prior(cbv_type=cbv_type, cbv_indices=cbv_indices, alpha=1.5e-1)
pltAxis = cbvcorrector.diagnose()
plt.show()


In [ ]:
cbvcorrector_lc = cbvcorrector.corrected_lc
cbvcorrector_lc.remove_outliers().plot()


In [ ]:
#Import the packages you need
from lightkurve.correctors import RegressionCorrector, DesignMatrix


In [ ]:
# Make a design matrix and pass it to a linear regression corrector
dm = DesignMatrix(S32.flux[:, ~aper_new], name='regressors')#.pca(5).append_constant()


In [ ]:
dm = dm.pca(10)
plt.plot(S32.time.value, dm.values + np.arange(10)*0.2, '.');